In [ ]:
import websockets
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## Arquivos esperados

Antes de rodar o servico no robo, coloque este arquivo diretamente dentro da pasta `Localizacao`:

- `camera_calibration_results.npz`: salvo pelo notebook `camera_calibration.ipynb`.

A mascara e construida automaticamente no servico a partir do mapa de reprojecao, mantendo apenas os pixels que apontam para dentro da imagem original e aplicando uma erosao para afastar as bordas.

In [ ]:
from pathlib import Path

LOCALIZACAO_DIR = Path.cwd().parent
CALIB_PATH = LOCALIZACAO_DIR / 'camera_calibration_results.npz'

print('CALIB_PATH:', CALIB_PATH)
print('existe:', CALIB_PATH.exists())

## Teste direto do detector

Rode esta celula no robo antes de iniciar o servidor. Ela inicializa o mesmo detector usado pelo servico, captura um quadro e confere se a saida tem o formato esperado.

In [ ]:
from picamera2 import Picamera2
import os

from servico_circulos import Detector, CAMERA_SIZE

Picamera2.set_logging(Picamera2.ERROR)
os.environ['LIBCAMERA_LOG_LEVELS'] = '3'

with Picamera2(tuning=os.environ.get('LIBCAMERA_RPI_TUNING_FILE', None)) as camera:
    camera.configure(camera.create_still_configuration(main={'size': CAMERA_SIZE}))
    camera.start()
    detector = Detector(9, 18, camera)
    coordenadas, timestamp = detector.detecta()
    camera.stop()

print('timestamp:', timestamp)
print('coordenadas:', coordenadas)
assert isinstance(coordenadas, list)
for p in coordenadas:
    assert len(p) == 2

## Teste pelo websocket

Em um terminal do robo, inicie o servico com:

```bash
python3 servico_circulos.py -p 8086
```

In [ ]:
endereco_robo="localhost" # Substitua pelo endereco do robo se necessario
porta_servico="8086"
async with websockets.connect(f"ws://{endereco_robo}:{porta_servico}/wsctrl") as websocket:
    await websocket.send("ack")
    res = json.loads(await  websocket.recv())

assert 'timestamp' in res
assert 'coordenadas' in res
print(res)

In [ ]:
fig, ax = plt.subplots()
ax.set_aspect('equal')
ax.set_xlim(0,400)
ax.set_ylim(-320,320)
for x, y in res["coordenadas"]:
    c = mpatches.Circle((x,y), radius=12)
    ax.add_patch(c)

### 8.4 - Filtro de partículas 

In [ ]:
import cv2
import numpy as np

# k-d tree do FLANN (algorithm=1). O mapa com as coordenadas dos circulos do
# ambiente e' indexado no matcher; calc_w usa knnMatch para achar, para cada
# observacao, os pontos mais proximos no mapa (passo 2 da secao 8.4.1).
FLANN_INDEX_KDTREE = 1


def cria_matcher(mapa):
    """Cria um cv2.FlannBasedMatcher ja carregado com o mapa do ambiente.

    `mapa` e' um array (M, 2) com as coordenadas globais (mm) de cada circulo
    conhecido do cenario.
    """
    mapa = np.asarray(mapa, dtype=np.float32).reshape(-1, 2)
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    matcher = cv2.FlannBasedMatcher(index_params, search_params)
    matcher.add([mapa])
    matcher.train()
    return matcher


def calc_w(estado, obs, knnmatcher, sigma2, alpha_r2):
    """Peso (verossimilhanca) de um estado do robo dado o conjunto de circulos
    observados, segundo o modelo sensorial da secao 8.4 (equacao 8.8).

    estado     : (3,)  -> (x, y, theta) global, em mm e rad.
    obs        : (N,2) -> coordenadas locais (mm) do centro de cada circulo.
    knnmatcher : cv2.FlannBasedMatcher carregado com o mapa (ver cria_matcher).
    sigma2     : sigma_c^2 da equacao 8.7 (constante).
    alpha_r2   : alpha (=> coeficiente de r^2) da equacao 8.7 (constante).
    """
    obs = np.asarray(obs, dtype=np.float32).reshape(-1, 2)
    n_obs = obs.shape[0]
    if n_obs == 0:
        return 0.0

    rx, ry, rth = float(estado[0]), float(estado[1]), float(estado[2])
    ct, st = np.cos(rth), np.sin(rth)

    # Passo 1: observacao (sistema local) -> coordenadas globais (c_x, c_y).
    #   c_x = r_x + o_x cos(theta) - o_y sin(theta)
    #   c_y = r_y + o_y cos(theta) + o_x sin(theta)
    c = np.empty((n_obs, 2), dtype=np.float64)
    c[:, 0] = rx + obs[:, 0] * ct - obs[:, 1] * st
    c[:, 1] = ry + obs[:, 0] * st + obs[:, 1] * ct

    # Mapa com as coordenadas dos circulos (recuperado do proprio matcher).
    mapa = np.asarray(knnmatcher.getTrainDescriptors()[0], dtype=np.float64)
    n_map = mapa.shape[0]
    k = min(n_obs, n_map)

    # Incerteza por observacao (eq. 8.7): sigma(r)^2 = sigma_c^2 + alpha * r^2,
    # com r^2 = o_x^2 + o_y^2 (objetos mais distantes -> mais incertos).
    r2 = obs[:, 0] ** 2 + obs[:, 1] ** 2
    sigma_r2 = sigma2 + alpha_r2 * r2

    # Passo 2: os k pontos do mapa mais proximos de cada observacao.
    matches = knnmatcher.knnMatch(c.astype(np.float32), k=k)

    # Passo 3: distancia ao quadrado normalizada pela incerteza, para cada
    # associacao candidata (observacao i, ponto j):
    #   d2_ij / (2 * sigma(r_i)^2)
    INF = np.inf
    cost = np.full((n_obs, n_map), INF)
    for i, candidatos in enumerate(matches):
        for m in candidatos:
            j = m.trainIdx
            d2 = (c[i, 0] - mapa[j, 0]) ** 2 + (c[i, 1] - mapa[j, 1]) ** 2
            cost[i, j] = d2 / (2.0 * sigma_r2[i])

    # Passo 4: associacao gulosa de maxima verossimilhanca. Enquanto houver
    # observacao nao associada: associe a observacao de MAIOR distancia minima
    # ao seu ponto mais proximo ainda livre e retire esse ponto dos demais.
    total = 0.0
    obs_livre = np.ones(n_obs, dtype=bool)
    pt_livre = np.ones(n_map, dtype=bool)
    for _ in range(k):
        melhor_pt = -np.ones(n_obs, dtype=int)
        melhor_custo = np.full(n_obs, INF)
        for i in range(n_obs):
            if not obs_livre[i]:
                continue
            linha = np.where(pt_livre, cost[i], INF)  # so' pontos ainda livres
            j = int(np.argmin(linha))
            if np.isfinite(linha[j]):
                melhor_custo[i] = linha[j]
                melhor_pt[i] = j
        # observacao com a MAIOR distancia minima entre as ainda livres
        candidato = melhor_custo.copy()
        candidato[~obs_livre] = -INF
        candidato[melhor_pt < 0] = -INF
        i_sel = int(np.argmax(candidato))
        if candidato[i_sel] == -INF:
            break  # nao ha mais associacoes possiveis
        j_sel = melhor_pt[i_sel]
        total += cost[i_sel, j_sel]
        obs_livre[i_sel] = False
        pt_livre[j_sel] = False

    # Peso total do estado (eq. 8.8). A normalizacao e' desnecessaria: o filtro
    # usa apenas valores proporcionais a probabilidade.
    return float(np.exp(-total))


#### Filtro de partículas (Localização de Monte Carlo)

Parâmetros do problema (seção 8.4):

- **20000** partículas iniciais, distribuição uniforme: `x, y` ∈ [-200, 200] mm e `θ` ∈ [0, 2π].
- `σ²_c = 20` (≈ 1/3 do raio do círculo ao quadrado + 2 px²) e `alpha = 0.01²`.
- Pesos calculados por `calc_w` (eq. 8.8) e **reamostragem** de **10000** partículas com `numpy.random.choice`.

As **observações** (`obs`) são as coordenadas locais, em mm, dos centros dos círculos detectados pelo serviço da seção 8.3. Inicie o serviço no robô e aponte a câmera para a folha A4 impressa do cenário (`drawing_1.pdf`, `drawing_s.pdf` ou `drawing_3.pdf`); cada célula de cenário lê um quadro pelo websocket. Veja `README_filtro_particulas.md` para o passo a passo no robô.


In [ ]:
# Constantes do filtro (seção 8.4)
N_PART = 20000          # populacao inicial
N_RESAMPLE = 10000      # populacao reamostrada
SIGMA2 = 20.0           # sigma_c^2  (eq. 8.7)
ALPHA_R2 = 0.01 ** 2    # alpha      (eq. 8.7)

ENDERECO_ROBO = "localhost"   # endereco do robo, se o notebook nao roda nele
PORTA_SERVICO = "8086"


async def obs_do_servico():
    """Le um quadro do servico de deteccao de circulos (secao 8.3) e devolve as
    coordenadas locais (mm) dos centros detectados como array (N, 2)."""
    uri = f"ws://{ENDERECO_ROBO}:{PORTA_SERVICO}/wsctrl"
    async with websockets.connect(uri) as websocket:
        await websocket.send("ack")
        res = json.loads(await websocket.recv())
    return np.asarray(res["coordenadas"], dtype=float).reshape(-1, 2)


def localiza(mapa, obs, n_part=N_PART, n_resample=N_RESAMPLE,
             sigma2=SIGMA2, alpha_r2=ALPHA_R2, rng=None):
    """Roda um passo de Localizacao de Monte Carlo para o 'robo sequestrado'.

    Inicializa `n_part` particulas uniformes, calcula os pesos com calc_w e
    reamostra `n_resample` particulas. Retorna (particulas_reamostradas, eta).
    """
    if rng is None:
        rng = np.random.default_rng()
    matcher = cria_matcher(mapa)

    # Populacao inicial uniforme (passo de previsao do robo sequestrado).
    part = np.empty((n_part, 3))
    part[:, 0] = rng.uniform(-200, 200, n_part)     # x
    part[:, 1] = rng.uniform(-200, 200, n_part)     # y
    part[:, 2] = rng.uniform(0, 2 * np.pi, n_part)  # theta

    # Passo de correcao: peso de cada particula.
    obs = np.asarray(obs, dtype=float).reshape(-1, 2)
    w = np.array([calc_w(part[i], obs, matcher, sigma2, alpha_r2)
                  for i in range(n_part)])
    eta = w.sum()
    if eta == 0.0:
        # eta ~ 0 sinaliza o "sequestro": nenhuma hipotese compativel.
        print("AVISO: eta = 0 (possivel sequestro / sem observacoes validas).")
        return part[:0], eta

    # Reamostragem proporcional aos pesos.
    idx = rng.choice(n_part, size=n_resample, p=w / eta)
    return part[idx], eta


def plota_resultado(mapa, part, ax, titulo):
    mapa = np.asarray(mapa, dtype=float).reshape(-1, 2)
    ax.set_aspect("equal")
    ax.set_xlim(-200, 200)
    ax.set_ylim(-200, 200)
    if len(part):
        ax.scatter(part[:, 0], part[:, 1], s=2, c="k", alpha=0.05,
                   label="partículas")
    ax.scatter(mapa[:, 0], mapa[:, 1], s=60, c="tab:blue", zorder=3,
               label="mapa")
    ax.set_title(titulo)
    ax.legend(loc="upper right", fontsize=8)


In [ ]:
# --- Cenário com 1 ponto de referência (Figuras 8.7 / 8.10) ---
# Aponte a câmera do robô para a folha drawing_1.pdf.
mapa_1 = np.array([[0.0, 0.0]])

obs = await obs_do_servico()
print("obs (local, mm):", obs)

part_1, eta_1 = localiza(mapa_1, obs)
print("eta =", eta_1)

fig, ax = plt.subplots(figsize=(5, 5))
plota_resultado(mapa_1, part_1, ax,
                "1 ponto de referência (anel de soluções)")
plt.show()


In [ ]:
# --- Cenário com 2 pontos de referência (Figuras 8.8 / 8.11) ---
# Aponte a câmera do robô para a folha drawing_s.pdf.
mapa_2 = np.array([[0.0, 297 / 6], [0.0, -297 / 6]])

obs = await obs_do_servico()
print("obs (local, mm):", obs)

part_2, eta_2 = localiza(mapa_2, obs)
print("eta =", eta_2)

fig, ax = plt.subplots(figsize=(5, 5))
plota_resultado(mapa_2, part_2, ax,
                "2 pontos de referência (duas nuvens / ambiguidade)")
plt.show()


In [ ]:
# --- Cenário com 3 pontos de referência (Figuras 8.9 / 8.12) ---
# Aponte a câmera do robô para a folha drawing_3.pdf.
mapa_3 = np.array([[0.0, 0.0], [105.0, 99.0], [105.0, -99.0]])

obs = await obs_do_servico()
print("obs (local, mm):", obs)

part_3, eta_3 = localiza(mapa_3, obs)
print("eta =", eta_3)

fig, ax = plt.subplots(figsize=(5, 5))
plota_resultado(mapa_3, part_3, ax,
                "3 pontos de referência (nuvem única)")
plt.show()
